In [1]:
import pandas as pd

df_artists = pd.read_csv('df_artists.csv')
df_songs   = pd.read_csv('df_songs.csv')

print(f"df_artists: {df_artists.shape}")
print(f"df_songs:   {df_songs.shape}")

df_artists: (14224, 54)
df_songs:   (38383, 27)


In [2]:
# Downloaded from https://github.com/gabminamedez/spotify-data/blob/master/data.csv
df_gabminamedez_spotify_data = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/gabminamedez_spotify_data.csv'
)

print(df_gabminamedez_spotify_data.shape)
print(df_gabminamedez_spotify_data.columns.tolist())
print()
print(df_gabminamedez_spotify_data.head(3).to_string())


(169909, 19)
['id', 'name', 'artists', 'duration_ms', 'release_date', 'year', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence', 'mode', 'key', 'popularity', 'explicit']

                       id                                      name                                   artists  duration_ms release_date  year  acousticness  danceability  energy  instrumentalness  liveness  loudness  speechiness    tempo  valence  mode  key  popularity  explicit
0  6KbQ3uYMLKb5jDxLF7wYDD               Singende Bataillone 1. Teil                       ['Carl Woitschach']       158648         1928  1928         0.995         0.708  0.1950             0.563    0.1510   -12.428       0.0506  118.469   0.7790     1   10           0         0
1  6KuQTIu1KoTTkLXKrwlLPV  Fantasiestücke, Op. 111: Più tosto lento  ['Robert Schumann', 'Vladimir Horowitz']       282133         1928  1928         0.994         0.379  0.0135             0.901    

In [3]:
# Cross check: df_songs vs df_gabminamedez_spotify_data
# Checks year coverage, match rate, and specific songs (e.g. The Weeknd's "Blinding Lights")
# that were missing from earlier versions of df_songs

import ast
import pandas as pd

# ── Spot-check specific songs ──────────────────────────────────
print("=" * 60)
print("SPOT CHECKS")
print("=" * 60)

checks = [
    ('Blinding Lights', 'The Weeknd'),
    ('Watermelon Sugar', 'Harry Styles'),
    ('Savage Love (Laxed - Siren Beat)', 'Jawsh 685'),
]

for title, artist in checks:
    in_songs = df_songs[df_songs['title'].str.lower() == title.lower()]
    in_spot  = df_gabminamedez_spotify_data[
        df_gabminamedez_spotify_data['name'].str.lower() == title.lower()
    ]
    print(f"\n'{title}' by {artist}:")
    print(f"  df_songs:  {len(in_songs)} row(s)")
    if len(in_songs):
        print(f"    → {in_songs[['title','performer_pre_normalized','first_charting_year']].to_string(index=False)}")
    print(f"  df_spot:   {len(in_spot)} row(s)")
    if len(in_spot):
        print(f"    → {in_spot[['name','artists','year','popularity']].to_string(index=False)}")

# ── Year coverage ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("YEAR COVERAGE")
print("=" * 60)
print(f"df_songs  first_charting_year: {int(df_songs['first_charting_year'].min())} – {int(df_songs['first_charting_year'].max())}")
print(f"df_spot   year:                {int(df_gabminamedez_spotify_data['year'].min())} – {int(df_gabminamedez_spotify_data['year'].max())}")
post2020 = (df_songs['first_charting_year'] > 2020).sum()
print(f"\ndf_songs rows after 2020 (outside df_spot coverage): {post2020:,} ({post2020/len(df_songs)*100:.1f}%)")

# ── Title-only match rate ──────────────────────────────────────
print("\n" + "=" * 60)
print("TITLE MATCH RATE (case-insensitive)")
print("=" * 60)
spot_titles = set(df_gabminamedez_spotify_data['name'].str.lower().dropna())
title_match = df_songs['title'].str.lower().isin(spot_titles)
print(f"All df_songs:            {title_match.sum():,} / {len(df_songs):,} ({title_match.mean()*100:.1f}%)")

df_pre2021 = df_songs[df_songs['first_charting_year'] <= 2020]
title_match_pre2021 = df_pre2021['title'].str.lower().isin(spot_titles)
print(f"df_songs up to 2020:     {title_match_pre2021.sum():,} / {len(df_pre2021):,} ({title_match_pre2021.mean()*100:.1f}%)")

# ── Title + artist match rate ──────────────────────────────────
print("\n" + "=" * 60)
print("TITLE + ARTIST MATCH RATE")
print("=" * 60)

# Parse artists column from stringified list to first artist name
def first_artist(val):
    try:
        return ast.literal_eval(val)[0].lower().strip()
    except Exception:
        return str(val).lower().strip()

df_spot_parsed = df_gabminamedez_spotify_data.copy()
df_spot_parsed['first_artist'] = df_spot_parsed['artists'].apply(first_artist)
df_spot_parsed['name_lower']   = df_spot_parsed['name'].str.lower()
spot_pairs = set(zip(df_spot_parsed['name_lower'], df_spot_parsed['first_artist']))

df_pre2021_check = df_pre2021.copy()
df_pre2021_check['title_lower']      = df_pre2021_check['title'].str.lower()
df_pre2021_check['performer_lower']  = df_pre2021_check['performer_pre_normalized'].str.lower().str.strip()
pair_match = df_pre2021_check.apply(
    lambda r: (r['title_lower'], r['performer_lower']) in spot_pairs, axis=1
)
print(f"df_songs up to 2020 (title + first artist): {pair_match.sum():,} / {len(df_pre2021_check):,} ({pair_match.mean()*100:.1f}%)")

# ── Unmatched sample ───────────────────────────────────────────
print("\n" + "=" * 60)
print("SAMPLE UNMATCHED (title-only, up to 2020)")
print("=" * 60)
unmatched = df_pre2021[~title_match_pre2021][['title','performer_pre_normalized','first_charting_year']]
print(unmatched.sample(min(10, len(unmatched))).to_string(index=False))


SPOT CHECKS

'Blinding Lights' by The Weeknd:
  df_songs:  1 row(s)
    →           title performer_pre_normalized  first_charting_year
Blinding Lights               The Weeknd                 2019
  df_spot:   2 row(s)
    →            name        artists  year  popularity
Blinding Lights ['The Weeknd']  2020          67
Blinding Lights ['The Weeknd']  2020         100

'Watermelon Sugar' by Harry Styles:
  df_songs:  1 row(s)
    →            title performer_pre_normalized  first_charting_year
Watermelon Sugar             Harry Styles                 2019
  df_spot:   2 row(s)
    →             name          artists  year  popularity
Watermelon Sugar ['Harry Styles']  2019          90
Watermelon Sugar ['Harry Styles']  2019          84

'Savage Love (Laxed - Siren Beat)' by Jawsh 685:
  df_songs:  1 row(s)
    →                            title performer_pre_normalized  first_charting_year
Savage Love (Laxed - Siren Beat) Jawsh 685 x Jason Derulo                 2020
  df_spot:   1 r

In [4]:
# Check Spotify match rate before merging — does not modify df_songs
import ast
import pandas as pd

# Deduplicate and explode df_spot (same logic as merge cell)
df_spot_dedup = (
    df_gabminamedez_spotify_data
    .sort_values('popularity', ascending=False)
    .drop_duplicates(subset='id')
    .copy()
)
df_spot_dedup['artists_parsed'] = df_spot_dedup['artists'].apply(
    lambda v: ast.literal_eval(v) if isinstance(v, str) else []
)
df_spot_exploded = df_spot_dedup.explode('artists_parsed').copy()
df_spot_exploded['title_lower']  = df_spot_exploded['name'].str.lower().str.strip()
df_spot_exploded['artist_lower'] = df_spot_exploded['artists_parsed'].str.lower().str.strip()
df_spot_exploded = (
    df_spot_exploded
    .sort_values('popularity', ascending=False)
    .drop_duplicates(subset=['title_lower', 'artist_lower'])
)

spot_pairs = set(zip(df_spot_exploded['title_lower'], df_spot_exploded['artist_lower']))

# Check match rate without touching df_songs
title_lower = df_songs['title'].str.lower().str.strip()
name_lower  = df_songs['name'].str.lower().str.strip()
matched     = pd.Series(list(zip(title_lower, name_lower))).isin(spot_pairs)

print("=" * 60)
print("MATCH RATE: df_songs → df_gabminamedez_spotify_data")
print("=" * 60)
print(f"All rows:           {matched.sum():,} / {len(df_songs):,} ({matched.mean()*100:.1f}%)")

mask_pre2021 = df_songs['first_charting_year'] <= 2020
print(f"Rows up to 2020:    {matched[mask_pre2021].sum():,} / {mask_pre2021.sum():,} ({matched[mask_pre2021].mean()*100:.1f}%)")
print(f"Rows after 2020:    {matched[~mask_pre2021].sum():,} / {(~mask_pre2021).sum():,} ({matched[~mask_pre2021].mean()*100:.1f}%)")

print("\nBy decade:")
df_check = df_songs.copy()
df_check['matched'] = matched.values
df_check['decade']  = (df_check['first_charting_year'] // 10 * 10).astype(int)
decade_stats = df_check.groupby('decade').apply(
    lambda g: pd.Series({
        'rows': len(g),
        'matched': g['matched'].sum(),
        'pct': f"{g['matched'].mean()*100:.1f}%"
    })
).reset_index()
print(decade_stats.to_string(index=False))


MATCH RATE: df_songs → df_gabminamedez_spotify_data
All rows:           13,219 / 38,383 (34.4%)
Rows up to 2020:    13,165 / 33,751 (39.0%)
Rows after 2020:    54 / 4,632 (1.2%)

By decade:
 decade  rows  matched   pct
   1950  1007      180 17.9%
   1960  7380     1635 22.2%
   1970  5589     1915 34.3%
   1980  4304     2052 47.7%
   1990  3822     1594 41.7%
   2000  4364     2324 53.3%
   2010  6319     3189 50.5%
   2020  5598      330  5.9%


/var/folders/1v/h2vyfq9j45n9_tmm_gg2s8tm0000gn/T/ipykernel_16513/2644414083.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  decade_stats = df_check.groupby('decade').apply(


In [5]:
# Merge Spotify audio features into df_songs
# Strategy:
#   1. Deduplicate df_spot by Spotify id (keep highest popularity snapshot)
#   2. Parse artists column from stringified list to Python list
#   3. Explode df_spot to one row per (song title, individual artist)
#   4. Merge with df_songs on title + name (df_songs.name = one individual artist)
# Songs after 2020 and songs not in Spotify's catalog will have NaN for audio features

import ast
import pandas as pd

SPOTIFY_COLS = [
    'id', 'duration_ms', 'release_date',
    'acousticness', 'danceability', 'energy', 'instrumentalness',
    'liveness', 'loudness', 'speechiness', 'tempo', 'valence',
    'mode', 'key', 'popularity', 'explicit'
]

# Step 1: Deduplicate df_spot by Spotify track id, keeping highest popularity snapshot
df_spot_dedup = (
    df_gabminamedez_spotify_data
    .sort_values('popularity', ascending=False)
    .drop_duplicates(subset='id')
    .copy()
)
print(f"df_spot after dedup by id: {len(df_spot_dedup):,} rows (was {len(df_gabminamedez_spotify_data):,})")

# Step 2: Parse artists from stringified list to Python list
df_spot_dedup['artists_parsed'] = df_spot_dedup['artists'].apply(
    lambda v: ast.literal_eval(v) if isinstance(v, str) else []
)

# Step 3: Explode to one row per (song, individual artist)
df_spot_exploded = df_spot_dedup.explode('artists_parsed').copy()
df_spot_exploded['title_lower']  = df_spot_exploded['name'].str.lower().str.strip()
df_spot_exploded['artist_lower'] = df_spot_exploded['artists_parsed'].str.lower().str.strip()

# Deduplicate exploded on (title, artist) in case of remaining duplicates — keep highest popularity
df_spot_exploded = (
    df_spot_exploded
    .sort_values('popularity', ascending=False)
    .drop_duplicates(subset=['title_lower', 'artist_lower'])
)
print(f"df_spot after explode + dedup: {len(df_spot_exploded):,} rows")

# Step 4: Build merge keys on df_songs
df_songs['title_lower'] = df_songs['title'].str.lower().str.strip()
df_songs['name_lower']  = df_songs['name'].str.lower().str.strip()

# Step 5: Merge
df_songs = df_songs.merge(
    df_spot_exploded[['title_lower', 'artist_lower'] + SPOTIFY_COLS].rename(
        columns={'id': 'spotify_id'}
    ),
    left_on=['title_lower', 'name_lower'],
    right_on=['title_lower', 'artist_lower'],
    how='left'
).drop(columns=['title_lower', 'name_lower', 'artist_lower'])

# Step 6: Coverage report
matched = df_songs['spotify_id'].notna().sum()
print(f"\nRows matched to Spotify: {matched:,} / {len(df_songs):,} ({matched/len(df_songs)*100:.1f}%)")

pre2021 = df_songs[df_songs['first_charting_year'] <= 2020]
matched_pre2021 = pre2021['spotify_id'].notna().sum()
print(f"Rows matched (up to 2020 only): {matched_pre2021:,} / {len(pre2021):,} ({matched_pre2021/len(pre2021)*100:.1f}%)")

# Spot check
print("\nSpot checks:")
for title in ['Blinding Lights', 'Watermelon Sugar', 'Savage Love (Laxed - Siren Beat)']:
    rows = df_songs[df_songs['title'].str.lower() == title.lower()][
        ['title', 'name', 'spotify_id', 'popularity', 'danceability', 'energy']
    ]
    print(f"\n'{title}':")
    print(rows.to_string(index=False))


df_spot after dedup by id: 169,909 rows (was 169,909)
df_spot after explode + dedup: 206,305 rows

Rows matched to Spotify: 13,219 / 38,383 (34.4%)
Rows matched (up to 2020 only): 13,165 / 33,751 (39.0%)

Spot checks:

'Blinding Lights':
          title       name             spotify_id  popularity  danceability  energy
Blinding Lights the weeknd 0VjIjW4GlUZAMYd2vXMi3b       100.0         0.514    0.73

'Watermelon Sugar':
           title         name             spotify_id  popularity  danceability  energy
Watermelon Sugar harry styles 6UelLqGlWMcVH1E5c4H7lY        90.0         0.548   0.816

'Savage Love (Laxed - Siren Beat)':
                           title                     name spotify_id  popularity  danceability  energy
Savage Love (Laxed - Siren Beat) jawsh 685 x jason derulo        NaN         NaN           NaN     NaN


In [6]:
# Check 25 random top-10 songs in df_songs for Spotify match and audio feature stats

sample = (
    df_songs[df_songs['peak_pos'] <= 10]
    .drop_duplicates(subset='title')
    .sample(25, random_state=42)
)

display_cols = [
    'title', 'performer_pre_normalized', 'first_charting_year', 'peak_pos',
    'spotify_id', 'popularity', 'danceability', 'energy', 'valence', 'tempo', 'explicit'
]

print(f"Top-10 songs sampled: 25")
matched = sample['spotify_id'].notna().sum()
print(f"Matched to Spotify:   {matched} / 25 ({matched/25*100:.0f}%)")
print()
print(sample[display_cols].to_string(index=False))


Top-10 songs sampled: 25
Matched to Spotify:   18 / 25 (72%)

                            title performer_pre_normalized  first_charting_year  peak_pos             spotify_id  popularity  danceability  energy  valence   tempo  explicit
                 Along Comes Mary          The Association                 1966         7 3Zje70mXjYyMuzge8j24L9        43.0         0.584   0.620    0.857 143.561       0.0
                How You Remind Me               Nickelback                 2001         1 0gmbgwZ8iqyMPmXefof8Yf        78.0         0.446   0.764    0.543 172.094       0.0
                          Missing  Everything But The Girl                 1995         2                    NaN         NaN           NaN     NaN      NaN     NaN       NaN
                       Bottoms Up               Trey Songz                 2010         6                    NaN         NaN           NaN     NaN      NaN     NaN       NaN
               Hate It Or Love It                 The Game          

In [8]:
# Save df_songs to CSV
df_songs.to_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_songs.csv',
    index=False
)

print(f"Saved df_songs: {len(df_songs):,} rows × {len(df_songs.columns)} columns")
print(f"\nColumns: {list(df_songs.columns)}")


Saved df_songs: 38,383 rows × 43 columns

Columns: ['title', 'name', 'performer_pre_normalized', 'original_performer_name', 'original_performer_name_normalized', 'peak_pos', 'wks_on_chart', 'first_charting_year', 'musicbrainz_artist_id', 'musicbrainz_recording_mbid', 'label', 'major_label', 'major_label_names', 'other_label', 'other_label_names', 'warner', 'universal', 'sony', 'label_mbid', 'song_genre_tags_musicbrainz', 'song_tags_musicbrainz', 'album_genre_tags_musicbrainz', 'song_major_genre_category_musicbrainz', 'album_major_genre_category_musicbrainz', 'aggregate_major_genre_category_musicbrainz', '#_of_genres_song', '#_of_genres_aggregate', 'spotify_id', 'duration_ms', 'release_date', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence', 'mode', 'key', 'popularity', 'explicit']
